In [1]:
import sys
sys.path.append('../')

from utils_basic import (
    copy_hamiltonian
)
from utils_ferm import (
    orthogonal_transform_obt_tbt,
    obt_phys_spatial_to_spin,
    tbt_phys_spatial_to_spin,
    make_short_H_ferm_op
)
from utils_states import (
    convert_TZ_format_to_sparse_format,
    convert_dense_format_to_sparse_format,
    tz_state_seniority_config,
    compress_state,
    decompress_state,
    create_composite_state
)
from utils_m1_seniority import (
    project_out_seniority_symmetries
)
from utils_m2_factorize import (
    expand_tensor_product,
    expand_tensor_product_for_incomplete_qubit_set,
    get_indices_mapping_2_wvn,
    factorize_state,
    evaluate_fully_classical_factors
)
from utils_m3_swap import (
    XorY_augment
)
from utils_m4_partitioning import (
    sorted_insertion_decomposition
)
from utils_results import (
    variance_of_decomp,
    sampling_cost
)
from openfermion import (
    get_sparse_operator,
    jordan_wigner
)

import numpy as np
import pickle


In [2]:
# load Q-SENSE basis states

molecule    = 'h2o'
bond_length = 3.0
filename    = f'../{molecule}_data/Uext_CSF_for_Praveen_Smik_{bond_length}.dump'

with open(filename, 'rb') as f:
    (
    list_list_refCSF,
    list_list_Uext_mp2_CSF,
    list_list_Uext_mp2_ampld,
    list_list_Uext_opt_ampld,
    list_orb_rot,
    x_orbrot,
    Enuc,
    obt_spatial,
    tbt_spatial
    ) = pickle.load(f)

# rotate orbitals and obtain Hamiltonian operator

if len(list_orb_rot) != 0:
    obt, tbt = orthogonal_transform_obt_tbt(x_orbrot,list_orb_rot,obt_spatial,tbt_spatial)
else:
    obt = obt_phys_spatial_to_spin(obt_spatial)
    tbt = tbt_phys_spatial_to_spin(tbt_spatial)

Hfer    = make_short_H_ferm_op(Enuc, obt, tbt)
Hqub    = jordan_wigner(Hfer)
Hsparse = get_sparse_operator(Hqub)

Nqubits = obt.shape[0]
Norb    = Nqubits // 2
dim     = 2 ** Nqubits

# obtain relevant information about Q-SENSE states (UCSFs, CSFs, W information) in a linear list

UCSF_tz_states = []
CSF_tz_states  = []
W_amplitudes   = []

for i, ucsf_list in enumerate(list_list_Uext_mp2_CSF):
    for j, ucsf in enumerate(ucsf_list):
        UCSF_tz_states.append(ucsf)
        CSF_tz_states.append(list_list_refCSF[i][j])
        W_amplitudes.append(list_list_Uext_mp2_ampld[i])

# process information so that we can taper and factorize the Q-SENSE states

Nstates          = len(UCSF_tz_states)
configs          = [tz_state_seniority_config(tz_state) for tz_state in UCSF_tz_states]
UCSF_information = [get_indices_mapping_2_wvn(CSF_tz_states[i], W_amplitudes[i], Norb) for i in range(Nstates)]

SW_list          = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'W']) for i in range(Nstates)]
SV_list          = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'V']) for i in range(Nstates)]
SN_list          = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'N']) for i in range(Nstates)]
state_type_list  = [UCSF_information[i][1] for i in range(Nstates)]

# taper and factorize the Q-SENSE basis states

statevectors                    = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in UCSF_tz_states]
tapered_statevectors            = [convert_dense_format_to_sparse_format(compress_state(psi.toarray()[0])) for psi in statevectors]
factorized_tapered_statevectors = [factorize_state(tapered_statevectors[i], SW_list[i], SV_list[i], SN_list[i], state_type_list[i]) 
                                   for i in range(Nstates)]

In [ ]:
#
#    Experiment 1: MSE metric with no modification of measurement procedure
#

assert molecule == 'h2o'

Hsub0       = np.zeros([Nstates, Nstates], dtype=np.complex128)
sig_matrix0 = np.zeros([Nstates, Nstates], dtype=np.complex128)

H = copy_hamiltonian(Hqub)
H -= H.constant
H.compress()
decomp = sorted_insertion_decomposition(H, 'fc')

for i in range(Nstates):
    print(f'{i, i}', end='\r')

    ket              = statevectors[i]
    Hsub0[i,i]       = (ket @ Hsparse @ ket.T)[0,0]
    var_metric       = variance_of_decomp(decomp, ket, Nqubits, general=True)
    sig_matrix0[i,i] = np.sqrt(var_metric)

Haug   = XorY_augment(H, Nqubits)
decomp = sorted_insertion_decomposition(Haug, 'fc')
for i in range(Nstates):
    for j in range(Nstates):
        if i > j:
            print(f'{i, j}', end='\r')

            bra              = statevectors[i]
            ket              = statevectors[j]
            Hsub0[i,j]       = (bra @ Hsparse @ ket.T)[0,0]
            Hsub0[j,i]       = (bra @ Hsparse @ ket.T)[0,0]

            comp             = create_composite_state(bra, ket, Nqubits)
            var_metric       = variance_of_decomp(decomp, comp, Nqubits + 1, general=True)
            sig_matrix0[i,j] = np.sqrt(var_metric)
            sig_matrix0[j,i] = np.sqrt(var_metric)

vals, vecs = np.linalg.eigh(Hsub0)
Egs        = vals[0]
c          = vecs[:,0]
cost       = sampling_cost(c, sig_matrix0)

print(f'''
    Final Results:
        Method              : {'PT'}
        Molecule            : {molecule}
        Bond Length         : {bond_length}
        Ground State Energy : {Egs}
        Sampling Cost       : {cost}
''')

In [ ]:
#
#    Experiment 2: MSE metric with qubit tapering (seniority based)
#

quantum_indices = [i for i in range(Nstates) if W_amplitudes[i] != []]

Hsub1       = np.zeros([Nstates, Nstates], dtype=np.complex128)
sig_matrix1 = np.zeros([Nstates, Nstates], dtype=np.complex128)

for i in range(Nstates):
    print(f'{i, i}', end='\r')

    ket_t      = tapered_statevectors[i]
    ket_config = configs[i]

    Htapered        = project_out_seniority_symmetries(Hqub, Nqubits, ket_config, ket_config)
    Htapered_sparse = get_sparse_operator(Htapered, Nqubits // 2)
    Hsub1[i,i]      = (ket_t @ Htapered_sparse @ ket_t.T)[0,0]

    if i in quantum_indices:
        Htapered        -= Htapered.constant
        Htapered.compress()
        decomp           = sorted_insertion_decomposition(Htapered, 'fc')
        var_metric       = variance_of_decomp(decomp, ket_t, Nqubits // 2, general=True)
        sig_matrix1[i,i] = np.sqrt(var_metric)

for i in range(Nstates):
    for j in range(Nstates):
        if i > j:
            print(f'{i, j}', end='\r')

            bra_t      = tapered_statevectors[i]
            bra_config = configs[i]

            ket_t      = tapered_statevectors[j]
            ket_config = configs[j]

            comp       = create_composite_state(bra_t, ket_t, Nqubits // 2)

            Htapered        = project_out_seniority_symmetries(Hqub, Nqubits, bra_config, ket_config)
            Htapered_sparse = get_sparse_operator(Htapered, Nqubits // 2)
            Hsub1[i,j]      = (bra_t @ Htapered_sparse @ ket_t.T)[0,0]
            Hsub1[j,i]      = (bra_t @ Htapered_sparse @ ket_t.T)[0,0]

            if (i in quantum_indices) or (j in quantum_indices):
                Htapered_aug     = XorY_augment(Htapered, Nqubits // 2)
                decomp           = sorted_insertion_decomposition(Htapered_aug, 'fc')
                var_metric       = variance_of_decomp(decomp, comp, Nqubits // 2 + 1, general=True)
                sig_matrix1[i,j] = np.sqrt(var_metric)
                sig_matrix1[j,i] = np.sqrt(var_metric)

vals, vecs = np.linalg.eigh(Hsub1)
Egs        = vals[0]
c          = vecs[:,0]
cost       = sampling_cost(c, sig_matrix1)

print(f'''
    Final Results:
        Method              : {'PT'}
        Molecule            : {molecule}
        Bond Length         : {bond_length}
        Ground State Energy : {Egs}
        Sampling Cost       : {cost}
''')

In [3]:
#
#    Experiment 3: MSE metric with qubit tapering and optimized constant shift (Q-SENSE v1 paper results)
#

quantum_indices = [i for i in range(Nstates) if W_amplitudes[i] != []]

Hsub2       = np.zeros([Nstates, Nstates], dtype=np.complex128)
sig_matrix2 = np.zeros([Nstates, Nstates], dtype=np.complex128)

for i in range(Nstates):
    print(f'{i, i}', end='\r')

    ket_t           = tapered_statevectors[i]
    ket_config      = configs[i]

    Htapered        = project_out_seniority_symmetries(Hqub, Nqubits, ket_config, ket_config)
    Htapered_sparse = get_sparse_operator(Htapered, Nqubits // 2)
    Hsub2[i,i]      = (ket_t @ Htapered_sparse @ ket_t.T)[0,0]

    if i in quantum_indices:
        Htapered        -= Htapered.constant
        Htapered.compress()
        decomp           = sorted_insertion_decomposition(Htapered, 'fc')
        var_metric       = variance_of_decomp(decomp, ket_t, Nqubits // 2, general=True)
        sig_matrix2[i,i] = np.sqrt(var_metric)
        
for i in range(Nstates):
    for j in range(Nstates):
        if i > j:
            print(f'{i, j}', end='\r')

            ij_shift        = 0.5 * (Hsub2[i,i] + Hsub2[j,j])

            bra_t           = tapered_statevectors[i]
            bra_config      = configs[i]

            ket_t           = tapered_statevectors[j]
            ket_config      = configs[j]

            comp_t          = create_composite_state(bra_t, ket_t, Nqubits // 2)

            Htapered        = project_out_seniority_symmetries(Hqub - ij_shift, Nqubits, bra_config, ket_config)
            Htapered_sparse = get_sparse_operator(Htapered, Nqubits // 2)
            Hsub2[i,j]      = (bra_t @ Htapered_sparse @ ket_t.T)[0,0]
            Hsub2[j,i]      = (bra_t @ Htapered_sparse @ ket_t.T)[0,0]

            if (i in quantum_indices) or (j in quantum_indices):
                Htapered_aug     = XorY_augment(Htapered, Nqubits // 2)
                decomp           = sorted_insertion_decomposition(Htapered_aug, 'fc')
                var_metric       = variance_of_decomp(decomp, comp_t, Nqubits // 2 + 1, general=True)
                sig_matrix2[i,j] = np.sqrt(var_metric)
                sig_matrix2[j,i] = np.sqrt(var_metric)

vals, vecs = np.linalg.eigh(Hsub2)
Egs        = vals[0]
c          = vecs[:,0]
cost       = sampling_cost(c, sig_matrix2)

print(f'''
    Final Results:
        Method              : {'PT'}
        Molecule            : {molecule}
        Bond Length         : {bond_length}
        Ground State Energy : {Egs}
        Sampling Cost       : {cost}
''')

(36, 35)
    Final Results:
        Method              : PT
        Molecule            : h2o
        Bond Length         : 2.0
        Ground State Energy : -74.76178857608413
        Sampling Cost       : (17.290841644074547+5.959150157201088e-08j)



In [3]:
#
#    Experiment 4: MSE metric using factorization and optimized constant shift (SOTA for v2 paper)
#

Hsub3       = np.zeros([Nstates, Nstates], dtype=np.complex128)
sig_matrix3 = np.zeros([Nstates, Nstates], dtype=np.complex128)

for i in range(Nstates):
    print(f'{i, i}', end='\r')

    ket_f      = factorized_tapered_statevectors[i]
    ket_labels = UCSF_information[i][0]
    ket_config = configs[i]

    Htapered        = project_out_seniority_symmetries(Hqub, Nqubits, ket_config, ket_config)
    HQ, ketQ, _, NQ = evaluate_fully_classical_factors(ket_f, ket_f, ket_labels, ket_labels, Htapered)

    if NQ == 0:
        Hsub3[i,i] = HQ.constant

    else:
        HQsparse   = get_sparse_operator(HQ)
        ketQ       = convert_dense_format_to_sparse_format(ketQ)
        Hsub3[i,i] = (ketQ @ HQsparse @ ketQ.T)[0,0]

        HQ              -= HQ.constant
        HQ.compress()
        decomp = sorted_insertion_decomposition(HQ, 'fc')
        var_metric       = variance_of_decomp(decomp, ketQ, NQ, general=True)
        sig_matrix3[i,i] = np.sqrt(var_metric)

for i in range(Nstates):
    for j in range(Nstates):
        if i > j:
            print(f'{i, j}', end='\r')

            ij_shift           = 0.5 * (Hsub3[i,i] + Hsub3[j,j])

            bra_f              = factorized_tapered_statevectors[i]
            bra_labels         = UCSF_information[i][0]
            bra_config         = configs[i]

            ket_f              = factorized_tapered_statevectors[j]
            ket_labels         = UCSF_information[j][0]
            ket_config         = configs[j]

            Htapered           = project_out_seniority_symmetries(Hqub - ij_shift, Nqubits, bra_config, ket_config)
            HQ, braQ, ketQ, NQ = evaluate_fully_classical_factors(bra_f, ket_f, bra_labels, ket_labels, Htapered)

            if NQ == 0:
                Hsub3[i,j] = HQ.constant
                Hsub3[j,i] = HQ.constant

            else:
                HQsparse         = get_sparse_operator(HQ, NQ)
                braQ             = convert_dense_format_to_sparse_format(braQ)
                ketQ             = convert_dense_format_to_sparse_format(ketQ)
                Hsub3[i,j]       = (braQ @ HQsparse @ ketQ.T)[0,0]
                Hsub3[j,i]       = (braQ @ HQsparse @ ketQ.T)[0,0]

                comp             = create_composite_state(braQ, ketQ, NQ)
                HQ_aug           = XorY_augment(HQ, NQ)
                decomp           = sorted_insertion_decomposition(HQ_aug, 'fc')
                var_metric       = variance_of_decomp(decomp, comp, NQ + 1, general=True)
                sig_matrix3[i,j] = np.sqrt(var_metric)
                sig_matrix3[j,i] = np.sqrt(var_metric)

vals, vecs = np.linalg.eigh(Hsub3)
Egs        = vals[0]
c          = vecs[:,0]
cost       = sampling_cost(c, sig_matrix3)

print(f'''
    Final Results:
        Method              : {'PT'}
        Molecule            : {molecule}
        Bond Length         : {bond_length}
        Ground State Energy : {Egs}
        Sampling Cost       : {cost}
''')

(35, 34)
    Final Results:
        Method              : PT
        Molecule            : h2o
        Bond Length         : 3.0
        Ground State Energy : -74.73773024878896
        Sampling Cost       : (0.00028877672956014746+2.1558638124483747e-17j)

